In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, T5EncoderModel

tokenizer = AutoTokenizer.from_pretrained("ai-forever/FRIDA")
model = T5EncoderModel.from_pretrained("ai-forever/FRIDA")



/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 219/219 [00:00<00:00, 16106.76it/s]


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)


In [4]:
def predict(texts, batch_size=4):
    texts = [f"categorize: {t}" for t in texts]
    text_emb = encode_batch(texts, batch_size=batch_size)
    scores = text_emb @ label_emb.T
    probs = torch.sigmoid(scores)
    return probs

In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("new_ds.csv")
df

,text,ASSORTMENT,PROMOTIONS,DELIVERY,PRICE,PRODUCTS_QUALITY,SUPPORT,CATALOG_NAVIGATION,PAYMENT
0,"Маленький выбор товаров, хотелось бы ассортиме...",1,0,0,0,0,0,0,0
1,Быстро,0,0,1,0,0,0,0,0
2,Доставка постоянно задерживается,0,0,1,0,0,0,0,0
3,Наценка и ассортимент расстраивают,1,0,0,1,0,0,0,0
4,Можно немного скинуть минимальную сумму заказа...,0,0,1,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...
2277,"Очень расстраивает, что сервис стал постоянно ...",0,0,1,0,0,0,0,0
2278,Хлеб привезли не свежий,0,0,0,0,1,0,0,0
2279,"Сервис испортился,доставка по [NUM],[NUM] часа(",0,0,1,0,0,0,0,0
2280,Последнее время постоянные проблемы с доставко...,0,0,1,0,0,0,0,0


In [7]:
label_texts = [
    "category: проблемы с доставкой, долгая доставка, курьер",
    "category: акции, скидки, промокоды",
    "category: ассортимент товаров, выбор",
    "category: высокая или низкая цена",
    "category: качество товара, брак",
    "category: служба поддержки, помощь клиенту",
    "category: навигация по каталогу, поиск товаров",
    "category: оплата, способы оплаты, проблемы с оплатой"
]

In [8]:
def pool(hidden_state, mask):
    s = torch.sum(hidden_state * mask.unsqueeze(-1).float(), dim=1)
    d = mask.sum(axis=1, keepdim=True).float()
    return s / d

def encode_batch(texts, batch_size=8):
    embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        tokens = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**tokens)
        emb = pool(outputs.last_hidden_state, tokens["attention_mask"])
        embs.append(F.normalize(emb, p=2, dim=1))
    return torch.cat(embs, dim=0).cpu().numpy()  # на CPU для sklearn

In [3]:
text=df["text"]
y_true=df.drop(columns=["text"])

In [4]:
from sklearn.model_selection import train_test_split


In [5]:
X_train_texts, X_val_texts, y_train, y_val = train_test_split(
    text, y_true, test_size=0.2, random_state=42
)

X_train_texts = X_train_texts.astype(str).tolist()
X_val_texts = X_val_texts.astype(str).tolist()

X_train = encode_batch(X_train_texts)
X_val = encode_batch(X_val_texts)


NameError: name 'encode_batch' is not defined

In [14]:
X_train_np = X_train
X_val_np = X_val

In [13]:
X_train_np = X_train.cpu().numpy() 
X_val_np = X_val.cpu().numpy()

AttributeError: 'numpy.ndarray' object has no attribute 'cpu'

In [16]:
import optuna
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score
import lightgbm as lgb

def objective(trial):
    # Гиперпараметры для базового классификатора
    param = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "num_leaves": trial.suggest_int("num_leaves", 20, 3000),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "class_weight": "balanced",
        "random_state": 42,
        "verbosity": -1
    }

    # Инициализация с новыми параметрами
    base_clf = lgb.LGBMClassifier(**param)
    clf = MultiOutputClassifier(base_clf)
    
    # Обучение
    clf.fit(X_train_np, y_train)
    
    # Валидация (используем macro f1 для мультилейбла)
    y_pred = clf.predict(X_val_np)
    score = f1_score(y_val, y_pred, average='macro')
    
    return score

# Запуск процесса
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print(f"Лучший F1: {study.best_value}")
print(f"Параметры: {study.best_params}")

[I 2026-04-01 20:47:58,739] A new study created in memory with name: no-name-3fb1de79-b0a4-41e4-a7aa-446cb9929366
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/h

Лучший F1: 0.6775279165400929
Параметры: {'n_estimators': 958, 'learning_rate': 0.033645187013767466, 'max_depth': 4, 'num_leaves': 2840, 'lambda_l1': 9.337010000559756, 'lambda_l2': 4.516508083666851, 'feature_fraction': 0.8168119561826707}


In [19]:
import json
best_params = study.best_params

with open('best_params.json', 'w', encoding='utf-8') as f:
    json.dump(best_params, f, indent=4, ensure_ascii=False)

In [ ]:
import lightgbm as lgb
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score


base_clf = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    class_weight='balanced',  # сбалансирует 0 и 1 автоматически
    random_state=42
)

clf = MultiOutputClassifier(base_clf)
# обучаем
clf.fit(X_train_np, y_train)




/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

In [6]:
from sentence_transformers import SentenceTransformer
import joblib
encoder=SentenceTransformer("/home/sasha/Python/VKR/pytorch_bert/model_weights/enc")
model=joblib.load("/home/sasha/Python/VKR/pytorch_bert/model_weights/classifier.joblib")
X_train_texts, X_val_texts, y_train, y_val = train_test_split(
    text, y_true, test_size=0.2, random_state=42
)

X_train_texts = X_train_texts.astype(str).tolist()
X_val_texts = X_val_texts.astype(str).tolist()

X_train = encoder.encode(X_train_texts)
X_val = encoder.encode(X_val_texts)


/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 219/219 [00:00<00:00, 16361.24it/s]


In [7]:
# предсказания
y_pred = model.predict(X_val)

/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

In [12]:
import numpy as np

y_probs = np.array([est.predict_proba(X_val)[:, 1] for est in model.estimators_]).T


/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

In [14]:
from sklearn.metrics import f1_score

In [15]:
y_pred_proba = model.predict_proba(X_val)  # список массивов для каждой метки
y_pred_thresh = np.zeros_like(y_val)

thresholds = [0.2, 0.01781436, 0.28073387, 0.04927171, 0.14049791, 0.02565653, 0.00265701, 0.00237321]  # подбираются эмпирически

for i, th in enumerate(thresholds):
    y_pred_thresh[:, i] = (y_pred_proba[i][:, 1] >= th).astype(int)

f1 = f1_score(y_val, y_pred_thresh, average='macro')
print(f"Macro F1 с порогами: {f1:.3f}")

Macro F1 с порогами: 0.000


/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

In [33]:
from scipy.optimize import differential_evolution
from sklearn.metrics import f1_score
import numpy as np



def macro_f1(thresholds):
    y_pred = np.zeros_like(y_val)
    for i, th in enumerate(thresholds):
        y_pred[:, i] = (y_pred_proba[i][:, 1] >= th).astype(int)
    return -f1_score(y_val, y_pred, average='macro') 

bounds = [(0, 1)] * len(y_pred_proba)
result = differential_evolution(macro_f1, bounds)
optimal_thresholds = result.x

# Применяем
y_pred_thresh = np.zeros_like(y_val)
for i, th in enumerate(optimal_thresholds):
    y_pred_thresh[:, i] = (y_pred_proba[i][:, 1] >= th).astype(int)

f1 = f1_score(y_val, y_pred_thresh, average='macro')
print("Максимальный Macro F1:", f1)
print("Порог для каждой метки:", optimal_thresholds)

Максимальный Macro F1: 0.7218905807372109
Порог для каждой метки: [0.09193241 0.09210526 0.79070408 0.04063725 0.19909005 0.06021983
 0.00679153 0.02501234]


In [34]:
from sklearn.metrics import classification_report
print(classification_report(y_val,y_pred_thresh))

              precision    recall  f1-score   support

           0       0.89      0.76      0.82        41
           1       1.00      0.43      0.60        14
           2       0.97      0.91      0.94       243
           3       0.86      0.87      0.86        84
           4       0.92      0.89      0.91        93
           5       0.81      0.68      0.74        57
           6       0.48      0.35      0.41        34
           7       1.00      0.33      0.50         9

   micro avg       0.90      0.82      0.86       575
   macro avg       0.87      0.65      0.72       575
weighted avg       0.89      0.82      0.85       575
 samples avg       0.74      0.71      0.71       575



/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, m

In [ ]:
from sklearn.metrics import hamming_loss

hloss = hamming_loss(y_val_np, y_pred_thresh)
print("Hamming Loss:", hloss)

Hamming Loss: 0.04431072210065645


In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_val,y_pred_thresh))

              precision    recall  f1-score   support

           0       0.91      0.76      0.83        41
           1       1.00      0.57      0.73        14
           2       0.91      0.95      0.93       243
           3       0.86      0.89      0.88        84
           4       0.93      0.90      0.92        93
           5       0.77      0.72      0.75        57
           6       0.42      0.38      0.40        34
           7       1.00      0.33      0.50         9

   micro avg       0.87      0.85      0.86       575
   macro avg       0.85      0.69      0.74       575
weighted avg       0.87      0.85      0.85       575
 samples avg       0.74      0.73      0.72       575



/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, m

In [ ]:
new_text_features = encode_batch(["молодцы!"]).cpu().numpy()

new_probs = clf.predict_proba(new_text_features)

new_pred = np.zeros((new_text_features.shape[0], len(thresholds)), dtype=int)
for i, th in enumerate(thresholds):
    new_pred[:, i] = (new_probs[i][:, 1] >= th).astype(int)

print(new_pred)

[[0 0 0 0 0 0 0 0]]


/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning